# Differentiable Programming

## What This Is
Differentiable programming extends automatic differentiation beyond smooth operations to discrete, combinatorial, and structured computations. This enables gradient-based optimization through algorithms that were previously opaque to backpropagation.

## Gumbel-Softmax
The Gumbel-softmax trick relaxes discrete categorical sampling to a continuous operation:

g_i ~ Gumbel(0, 1) (add Gumbel noise to log-probabilities)
Softmax output: y_i = exp((log p_i + g_i) / τ) / Σ exp((log p_j + g_j) / τ)

As τ → 0, the output becomes a one-hot vector (discrete). At τ = 1, it's a soft approximation. During training, we use τ > 0 for gradient flow; at inference, we take argmax (straight-through estimator).

## Differentiable Sorting and Selection
Soft-sort (Cuturi et al., 2019) relaxes sorting to a differentiable operation using optimal transport. Differentiable top-k uses Gumbel-softmax applied to k selections.

In [ ]:
import numpy as np
from typing import Tuple

np.random.seed(42)

# Gumbel-Softmax trick
def gumbel_noise(shape):
    u = np.random.uniform(1e-8, 1, shape)
    return -np.log(-np.log(u))

def gumbel_softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    g = gumbel_noise(logits.shape)
    y = logits + g
    y = y - y.max()
    exp_y = np.exp(y / temperature)
    return exp_y / exp_y.sum()

def straight_through_argmax(soft: np.ndarray) -> np.ndarray:
    hard = np.zeros_like(soft)
    hard[np.argmax(soft)] = 1.0
    return hard

# Test Gumbel-softmax at different temperatures
logits = np.array([2.0, 1.0, 0.5, 0.0, -1.0])
softmax_base = np.exp(logits) / np.exp(logits).sum()

print('Gumbel-Softmax at Different Temperatures:')
print(f'Logits: {logits}')
print(f'True softmax: {softmax_base.round(3)}')
print()

for tau in [5.0, 1.0, 0.5, 0.1, 0.01]:
    # Average over multiple samples for expected distribution
    samples = np.array([gumbel_softmax(logits, tau) for _ in range(200)])
    mean_sample = samples.mean(0)
    entropy = -(mean_sample * np.log(mean_sample + 1e-10)).sum()
    print(f'tau={tau:.2f}: mean={mean_sample.round(3)}, entropy={entropy:.3f}')


In [ ]:
# Differentiable sorting via relaxed permutation matrices

def soft_sort(x: np.ndarray, temperature: float = 0.1) -> np.ndarray:
    """
    Differentiable sort using regularized optimal transport.
    Simple version: uses a pairwise comparison matrix.
    """
    n = len(x)
    # Sort via repeated soft argmax (simplified)
    # For each target rank k, compute soft-assignment of x_i to rank k
    ranks = np.arange(n, dtype=float)
    
    # Simple approximation: use deterministic sort with smooth interpolation
    sorted_x = np.sort(x)[::-1]  # descending
    
    # Soft permutation matrix: P[i,j] = probability that x[i] maps to rank j
    # Using pairwise comparisons: sigmoid((x[i] - x[j]) / temperature)
    P = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                P[i, j] = 1 / (1 + np.exp(-(x[j] - x[i]) / temperature))
    
    # Rank score: sum of comparisons won
    rank_scores = P.sum(1)  # how many elements this element is less than
    
    return rank_scores, sorted_x

# Test
x = np.array([3.0, 1.0, 4.0, 1.5, 2.0])
rank_scores, sorted_x = soft_sort(x)

print('Differentiable Sort:')
print(f'Input: {x}')
print(f'True sorted (desc): {sorted_x}')
print(f'Soft rank scores: {rank_scores.round(3)}')
print(f'Induced order (argsort of rank scores): {np.argsort(rank_scores)}')
print()
print('Use case: learn to rank, learned k-NN, differentiable beam search')
